# Exp 6 🏆 — MPNet + W&B Sweep

In [1]:
!pip install datasets wandb scikit-learn sentence-transformers gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 29.1 MB/s eta 0:00:00


In [2]:
import torch, torch.nn as nn, torch.optim as optim, torch.backends.cudnn as cudnn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score
from datasets import load_dataset
import numpy as np, copy, wandb
SEED=42
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
cudnn.benchmark=False; cudnn.deterministic=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device:{device}')

Device:cuda


In [3]:
data=load_dataset('Sp1786/multiclass-sentiment-analysis-dataset')
def remove_empty(row):
    return all(row[f] not in [None,''] for f in ['id','text','label','sentiment'])
train_data=data['train'].filter(remove_empty)
dev_data=data['validation'].filter(remove_empty)
test_data=data['test'].filter(remove_empty)
output_size=len(set(train_data['label']))
train_labels=train_data['label']
test_labels_list=test_data['label']
print(f'Train:{len(train_data)}|Dev:{len(dev_data)}|Test:{len(test_data)}|Classes:{output_size}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train_df.csv: 0.00B [00:00, ?B/s]

val_df.csv: 0.00B [00:00, ?B/s]

test_df.csv: 0.00B [00:00, ?B/s]

...

Generating test split:   0%|          | 0/5206 [00:00<?, ? examples/s]

Filter:   0%|          | 0/31232 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5205 [00:00<?, ? examples/s]

Filter:   0%|          | 0/5206 [00:00<?, ? examples/s]

Train:31232|Dev:5205|Test:5205|Classes:3


In [4]:
from sentence_transformers import SentenceTransformer
model_emb=SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
train_np=model_emb.encode(train_data['text'],batch_size=64,show_progress_bar=True,convert_to_numpy=True)
dev_np=model_emb.encode(dev_data['text'],batch_size=64,show_progress_bar=True,convert_to_numpy=True)
test_np=model_emb.encode(test_data['text'],batch_size=64,show_progress_bar=True,convert_to_numpy=True)
input_size=768
train_t=torch.FloatTensor(train_np).to(device)
dev_t=torch.FloatTensor(dev_np).to(device)
test_t=torch.FloatTensor(test_np).to(device)
dev_labels_t=torch.tensor(dev_data['label'],dtype=torch.long).to(device)
print(f'MPNet:{train_t.shape}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/488 [00:00<?, ?it/s]

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

Batches:   0%|          | 0/82 [00:00<?, ?it/s]

MPNet:torch.Size([31232, 768])


In [5]:
class MLP(nn.Module):
    def __init__(self, i, h, o, d=0.0):
        super().__init__()
        self.fc1=nn.Linear(i,h)
        self.fc2=nn.Linear(h,h//2)
        self.fc3=nn.Linear(h//2,o)
        self.activation=nn.GELU()
        self.output_act=nn.Softmax(dim=1)
        self.dropout=nn.Dropout(p=d)
    def forward(self,x):
        x=self.dropout(self.activation(self.fc1(x)))
        x=self.dropout(self.activation(self.fc2(x)))
        return self.output_act(self.fc3(x))

In [6]:
def make_sweep_fn(train_t,dev_t,dev_l,test_t,inp,lbl):
    def train_fn():
        with wandb.init() as run:
            cfg=run.config
            torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
            model=MLP(inp,cfg.hidden_size,output_size,cfg.dropout).to(device)
            opt=optim.Adam(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)
            lfn=nn.CrossEntropyLoss()
            best_dev,best_state=0,None
            for epoch in range(cfg.num_epochs):
                model.train()
                eloss=0
                for i in range(0,len(train_t),cfg.batch_size):
                    bd=train_t[i:i+cfg.batch_size]
                    bl=torch.tensor(train_labels[i:i+cfg.batch_size],device=device)
                    out=model(bd)
                    loss=lfn(out,bl)
                    opt.zero_grad(); loss.backward(); opt.step()
                    eloss+=loss.item()
                model.eval()
                with torch.no_grad():
                    da=(torch.argmax(model(dev_t),dim=1)==dev_l).float().mean().item()
                if da>best_dev:
                    best_dev=da
                    best_state=copy.deepcopy(model.state_dict())
                wandb.log({'epoch':epoch+1,'dev_accuracy':da,'best_dev_accuracy':best_dev,'train_loss':eloss/len(train_t)})
            model.load_state_dict(best_state)
            model.eval()
            with torch.no_grad():
                tp=torch.argmax(model(test_t),dim=1)
                ta=accuracy_score(test_labels_list,tp.cpu().tolist())
            wandb.log({'test_accuracy':ta})
            print(f'[{lbl}] Dev:{best_dev:.4f}|Test:{ta*100:.2f}%')
    return train_fn

SWEEP_CFG={'method':'bayes','metric':{'name':'best_dev_accuracy','goal':'maximize'},
'parameters':{'learning_rate':{'distribution':'log_uniform_values','min':1e-5,'max':1e-2},
'hidden_size':{'values':[256,512,1000,2000]},'dropout':{'values':[0.0,0.1,0.2,0.3,0.5]},
'weight_decay':{'values':[0.0,1e-5,1e-4,1e-3]},'num_epochs':{'values':[20,30,50]},
'batch_size':{'values':[64,128,256]}}}

In [7]:
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aileen02-ko (imeanseo_) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [8]:
sweep_id=wandb.sweep({**SWEEP_CFG,'name':'exp6-mpnet'},project='nlp-hw1')
print(f'Sweep ID:{sweep_id}')
train_fn=make_sweep_fn(train_t,dev_t,dev_labels_t,test_t,input_size,'Exp6-MPNet')
wandb.agent(sweep_id,function=train_fn,count=20)

Create sweep with ID: v6x7gb20
Sweep URL: https://wandb.ai/imeanseo_/nlp-hw1/sweeps/v6x7gb20
Sweep ID:v6x7gb20


wandb: Agent Starting Run: je1b5bke with config:
wandb: 	batch_size: 128
wandb: 	dropout: 0.1
wandb: 	hidden_size: 2000
wandb: 	learning_rate: 0.0004984119130529279
wandb: 	num_epochs: 30
wandb: 	weight_decay: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp6-MPNet] Dev:0.6676|Test:67.03%


best_dev_accuracy,▁▄▅▆▆▆▇▇▇▇████████████████████
dev_accuracy,▁▄▅▆▆▆▇▇▇▇█▇▇▇▇██▇▇▇█▇██▇▇█▇▇▇
epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
test_accuracy,▁
train_loss,█▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.66763
dev_accuracy,0.66647
epoch,30
test_accuracy,0.67032
train_loss,0.00686


wandb: Agent Starting Run: cgc5qf6u with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.3
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 8.769566057017218e-05
wandb: 	num_epochs: 20
wandb: 	weight_decay: 1e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

[Exp6-MPNet] Dev:0.6734|Test:67.40%


best_dev_accuracy,▁▁▃▃▃▅▅▅▆▆▆▆▇▇▇█████████████████████████
dev_accuracy,▁▁▃▃▃▄▅▄▅▆▆▆▆▇▆█▇▆▇█▇█▇█▅▇▆▇▇▇█▇▇▅▇▆▅▇▇▅
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
test_accuracy,▁
train_loss,█▆▆▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▁▂▂▂▁▂▁▁▁▁▁▁
best_dev_accuracy,0.67339
dev_accuracy,0.66302
epoch,50
test_accuracy,0.67397
train_loss,0.00325


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: g92ctvne with config:
wandb: 	batch_size: 256
wandb: 	dropout: 0.2
wandb: 	hidden_size: 1000
wandb: 	learning_rate: 0.0005444387268944882
wandb: 	num_epochs: 50
wandb: 	weight_decay: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


[Exp6-MPNet] Dev:0.6720|Test:67.34%


best_dev_accuracy,▁▄▅▆▆▆▆▆▇▇▇█████████████████████████████
dev_accuracy,▁▄▅▆▅▅▆▅▇▆▇█▇▆▇█▇▇█▆▆▅▆▆▆▆▆▆▆▆▄▅▄▅▅▃▄▅▄▆
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
test_accuracy,▁
train_loss,█▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
best_dev_accuracy,0.67205
dev_accuracy,0.66763
epoch,50
test_accuracy,0.67339
train_loss,0.00328


In [11]:
USERNAME='imeanseo_'
api=wandb.Api()
sw=api.sweep(f'{USERNAME}/nlp-hw1/{sweep_id}')
best=sw.best_run()
print('\n'+'='*60)
print('🏆 Best Run Config:')
for k,v in dict(best.config).items():
    print(f'  {k:<20}: {v}')
print(f"\nBest Dev:{best.summary['best_dev_accuracy']:.4f}")
print(f"Test:{best.summary['test_accuracy']*100:.2f}%")
print('='*60)

wandb: Sorting runs by -summary_metrics.best_dev_accuracy



🏆 Best Run Config:
  dropout             : 0.1
  batch_size          : 128
  num_epochs          : 50
  hidden_size         : 512
  weight_decay        : 1e-05
  learning_rate       : 0.003965933641576531

Best Dev:0.6788
Test:66.28%


## Best Config로 재학습 + 저장

In [ ]:
# ⚠️ 위 출력 값으로 수정
BEST_H,BEST_LR,BEST_D,BEST_WD,BEST_EP,BEST_BS=1000,0.0003,0.2,1e-4,30,128
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
final=MLP(input_size,BEST_H,output_size,BEST_D).to(device)
opt=optim.Adam(final.parameters(),lr=BEST_LR,weight_decay=BEST_WD)
lfn=nn.CrossEntropyLoss()
best_dev,best_state=0,None
for epoch in range(BEST_EP):
    final.train()
    for i in range(0,len(train_t),BEST_BS):
        bd=train_t[i:i+BEST_BS]
        bl=torch.tensor(train_labels[i:i+BEST_BS],device=device)
        loss=lfn(final(bd),bl)
        opt.zero_grad(); loss.backward(); opt.step()
    final.eval()
    with torch.no_grad():
        da=(torch.argmax(final(dev_t),dim=1)==dev_labels_t).float().mean().item()
    if da>best_dev:
        best_dev,best_state=da,copy.deepcopy(final.state_dict())
    print(f'Epoch {epoch+1}/{BEST_EP}|Dev:{da:.4f}')
final.load_state_dict(best_state)
torch.save(best_state,'best_model_exp6.pt')
with torch.no_grad():
    test_acc=accuracy_score(test_labels_list,torch.argmax(final(test_t),dim=1).cpu().tolist())
print(f'\n✅ 저장:best_model_exp6.pt|Dev:{best_dev:.4f}|Test:{test_acc*100:.2f}%')

In [ ]:
from google.colab import files
files.download('best_model_exp6.pt')